<a href="https://colab.research.google.com/github/dkang1630/Conductor_Image_Classification/blob/main/U_Net_data_preparer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import os
from google.colab import drive
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Google Drive paths
base_path = "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net"
SAM_base_path = os.path.join(base_path, "SAM")
Binary_masked_base_path = os.path.join(base_path, "Binary_Masked")

In [ ]:
#Function to convert SAM-segmented image to binary mask

def convert_to_binary_mask(input_image_path, output_mask_path, gray_background=(100, 100, 100), tolerance=10):
    # Load the input image
    image = cv2.imread(input_image_path, cv2.IMREAD_COLOR)

    if image is None:
        print(f"Failed to load image from {input_image_path}")
        return False

    # Convert to RGB (original color space)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Gray background detection
    lower_bound = np.array([gray_background[0] - tolerance,
                           gray_background[1] - tolerance,
                           gray_background[2] - tolerance])
    upper_bound = np.array([gray_background[0] + tolerance,
                           gray_background[1] + tolerance,
                           gray_background[2] + tolerance])

    # Create background mask
    background_mask = cv2.inRange(image_rgb, lower_bound, upper_bound)

    # Invert to get foreground mask
    binary_mask = cv2.bitwise_not(background_mask) // 255

    # Convert to 8-bit format
    binary_mask = (binary_mask * 255).astype(np.uint8)

    # --- Original Morphological Operations Preserved ---
    # Large elliptical kernel for dilation/erosion
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))

    # Dilation first to expand regions
    dilated_mask = cv2.dilate(binary_mask, kernel, iterations=1)

    # Erosion to shrink back while preserving expanded boundaries
    eroded_mask = cv2.erode(dilated_mask, kernel, iterations=1)

    # Save result
    cv2.imwrite(output_mask_path, eroded_mask)
    return True


In [ ]:
for img in os.listdir(SAM_base_path):
    if img.endswith('.jpg'):
      cleaned_name = img.replace("Copy of ", "")

      input_img_path = os.path.join(SAM_base_path, img)
      output_mask_path = os.path.join(Binary_masked_base_path, f"mask_{cleaned_name}")

      # Convert image to binary mask
      success = convert_to_binary_mask(input_img_path, output_mask_path)
      if success:
        print(f"Saved binary mask: {output_mask_path}")
      else:
        print(f"Skipping {input_img_path} (Failed to process)")

print("Processing and saving completed!")

Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_1.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_4.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_7.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_10.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_13.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_16.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mask_img_19.jpg
Saved binary mask: /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Binary_Masked/mas